# Mi grimorio de visión por computadora
**AulaVisión · Grimoire Lab — Black Clover Edition**

Semanas 12 y 13 · 2 al 13 de noviembre de 2026.

Ejecuta las celdas en orden. Primero observarás listas, condicionales y matrices; después aplicarás filtros a imágenes reales y probarás formularios web. Las ilustraciones provienen del sitio oficial de Black Clover y pertenecen a sus titulares. Este es un proyecto académico de fans no oficial.

## Preparación
En un entorno nuevo, instala las bibliotecas con la siguiente celda. Necesitas conexión a Internet para la instalación y la descarga de las ilustraciones.

In [ ]:
%pip install numpy opencv-python-headless matplotlib Pillow

In [ ]:
import io
import urllib.request
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print('Entorno listo')

## Misión 1 — Listas, diccionarios y bucles
Un píxel de gris guarda una intensidad entre 0 y 255. Recorre la lista y observa la clasificación.

In [ ]:
brillos = [20, 100, 160, 250]
foto = {"nombre": "Asta", "ancho": 900, "alto": 560}
for pixel in brillos:
    estado = "CLARO" if pixel >= 128 else "OSCURO"
    print(pixel, estado)
print(foto["nombre"], foto["ancho"], "×", foto["alto"])

**Prueba tú:** añade los valores 0, 128 y 255 a la lista. ¿Cómo se clasifican?

## Misión 2 — Condicionales y umbral
Modifica las dos variables y ejecuta. El ejercicio clasifica como claro el valor mayor o igual al umbral.

In [ ]:
pixel = 160
umbral = 128
if not 0 <= pixel <= 255 or not 0 <= umbral <= 255:
    raise ValueError("Usa intensidades enteras entre 0 y 255")
print("CLARO" if pixel >= umbral else "OSCURO")

## Misión 3 — Cargar imágenes y leer matrices
Elige `asta`, `yuno` o `noelle`. La transparencia se compone sobre fondo blanco. OpenCV representa la imagen como una matriz BGR.

In [ ]:
personaje = "asta"
archivos = {"asta": "asta-action.png", "yuno": "yuno.png", "noelle": "noelle.png"}
if personaje not in archivos:
    raise ValueError("Elige asta, yuno o noelle")
url = "https://raw.githubusercontent.com/Loren07-kie/Rev.cuad/main/static/images/" + archivos[personaje]
with urllib.request.urlopen(url, timeout=30) as respuesta:
    fuente = Image.open(io.BytesIO(respuesta.read())).convert("RGBA")
fondo = Image.new("RGBA", fuente.size, (255, 255, 255, 255))
rgb = np.array(Image.alpha_composite(fondo, fuente).convert("RGB"))
imagen = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
print(imagen.shape)
print(imagen.dtype)
plt.figure(figsize=(8, 5))
plt.imshow(rgb)
plt.axis("off");

## Misión 4 — Filtros con OpenCV
Compara los resultados. No se está entrenando un modelo de IA: son transformaciones deterministas de imagen.

In [ ]:
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
gaussiano = cv2.GaussianBlur(imagen, (15, 15), 0)
suave = cv2.GaussianBlur(gris, (5, 5), 0)
bordes = cv2.Canny(suave, 100, 200)
umbral = 128
_, binaria = cv2.threshold(gris, umbral, 255, cv2.THRESH_BINARY)
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, titulo, resultado in zip(axes, ["Grises", "Gaussiano", "Canny", "Umbral"], [gris, cv2.cvtColor(gaussiano, cv2.COLOR_BGR2RGB), bordes, binaria]):
    ax.imshow(resultado, cmap="gray", vmin=0, vmax=255)
    ax.set_title(titulo)
    ax.axis("off")
plt.tight_layout()

**Observa:** `cv2.threshold` asigna blanco para valores estrictamente mayores al umbral. El ejercicio anterior de condicionales usa `>=`. Comprueba el caso exacto 128.

In [ ]:
muestra = np.array([[127, 128, 129]], dtype=np.uint8)
print("Condicional >= 128:", muestra >= 128)
print("OpenCV > 128:", cv2.threshold(muestra, 128, 255, cv2.THRESH_BINARY)[1])

### Histograma y un píxel
El histograma resume la distribución de intensidades. Modifica `x` e `y` para explorar un punto.

In [ ]:
print("Brillo medio:", round(float(gris.mean()), 1))
print("Porcentaje claro (>= umbral):", round(float((gris >= umbral).mean()) * 100, 1))
x, y = imagen.shape[1] // 2, imagen.shape[0] // 2
print("Píxel RGB en", (x, y), ":", rgb[y, x])
plt.hist(gris.ravel(), bins=16, range=(0, 256), color="#759728")
plt.xlabel("Intensidad de gris")
plt.ylabel("Número de píxeles")
plt.show()

## Misión 5 — De Java a una solicitud web
El proyecto original de la semana 13 utiliza Java, Spring Boot, `@PostMapping` y `@RequestParam` para saludo, registro y suma. En esta integración usamos Flask para las rutas web. El siguiente bloque es una referencia Java, no una celda Python ejecutable.

```java
@PostMapping("/resultado")
public String resultado(@RequestParam("num1") double num1,
                        @RequestParam("num2") double num2) {
    return "La suma es: " + (num1 + num2);
}
```

La celda Python siguiente envía datos de prueba a la aplicación publicada.

In [ ]:
import json
base_url = "https://aulavision-lab-lorena.onrender.com"
datos = {"a": 12, "b": 13, "operacion": "suma"}
solicitud = urllib.request.Request(base_url + "/api/calcular",
    data=json.dumps(datos).encode(), headers={"Content-Type": "application/json"})
with urllib.request.urlopen(solicitud, timeout=90) as respuesta:
    print(json.load(respuesta))

## Misión 6 — Reto final
1. Cambia el personaje y repite los filtros.
2. Compara dos umbrales y explica por qué cambian las zonas blancas.
3. Cambia el tamaño del kernel gaussiano a `(31, 31)`.
4. Haz una división entre cero con la API e interpreta la respuesta de validación.
5. Explica la diferencia entre un control de formulario y el código que procesa sus datos.

### Fuentes visuales
- [Black Clover — sitio oficial](https://bclover.jp/season1/)
- [Personajes oficiales](https://bclover.jp/character/)

Proyecto académico de Lorena Abigail Coca Ccorpa. Logo propio generado con IA; personajes originales de Black Clover.